In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

In [2]:
IN_KAGGLE = os.path.exists('/kaggle/input') 

if IN_KAGGLE :
    # change that to name of folder on kaggle
    DATA_DIR_TRAIN = Path('/kaggle/input/house-prices-advanced-regression-techniques')
    DATA_DIR_TEST = Path('/kaggle/input/house-prices-advanced-regression-techniques')

else :
    # Path.cwd() <> current working directory
    # parent() <> undo a folder 
    DATA_DIR_TRAIN = Path.cwd().parent / 'datasets'  / 'processed' 
    DATA_DIR_TEST = Path.cwd().parent / 'datasets'  / 'raw' 

train_data = pd.read_csv( DATA_DIR_TRAIN / 'cleaned_train_data.csv')
test_data = pd.read_csv( DATA_DIR_TEST / 'test.csv')
test_data.shape

(1459, 80)

In [3]:
def summary(df):
    pd.set_option('display.max.rows' , None)
    pd.set_option('display.max.columns' , None)

    summary_df = pd.DataFrame({ 'Dtype': df.dtypes , 
                               'Count_Null' : df.isnull().sum() , 
                               'Null_Percent' : ( df.isnull().sum()  / len(df)) * 100 ,
                               'Unique' : df.nunique()
                              })

    return summary_df

summary(test_data)

,Dtype,Count_Null,Null_Percent,Unique
Id,int64,0,0.000000,1459
MSSubClass,int64,0,0.000000,16
MSZoning,str,4,0.274160,5
LotFrontage,float64,227,15.558602,115
LotArea,int64,0,0.000000,1106
Street,str,0,0.000000,2
Alley,str,1352,92.666210,2
LotShape,str,0,0.000000,4
LandContour,str,0,0.000000,4
Utilities,str,2,0.137080,1


### Nulls

In [4]:
test_data = test_data.drop(
['Alley' , 'Fence' , 'MiscFeature' , 'MasVnrType', 'FireplaceQu'] , axis= 1)

test_data.shape

(1459, 75)

In [5]:
nulls = test_data.isnull().sum()
nulls = nulls[ nulls > 0 ]
nulls

MSZoning           4
LotFrontage      227
Utilities          2
Exterior1st        1
Exterior2nd        1
MasVnrArea        15
BsmtQual          44
BsmtCond          45
BsmtExposure      44
BsmtFinType1      42
BsmtFinSF1         1
BsmtFinType2      42
BsmtFinSF2         1
BsmtUnfSF          1
TotalBsmtSF        1
BsmtFullBath       2
BsmtHalfBath       2
KitchenQual        1
Functional         2
GarageType        76
GarageYrBlt       78
GarageFinish      78
GarageCars         1
GarageArea         1
GarageQual        78
GarageCond        78
PoolQC          1456
SaleType           1
dtype: int64

##### Handling with Categorical Columns

In [6]:
cat_cols_nulls = test_data[nulls.index].select_dtypes(
     include=['string' , 'object']).isnull().sum()

cat_cols_nulls = cat_cols_nulls[ cat_cols_nulls > 0 ]   
cat_cols_nulls

MSZoning           4
Utilities          2
Exterior1st        1
Exterior2nd        1
BsmtQual          44
BsmtCond          45
BsmtExposure      44
BsmtFinType1      42
BsmtFinType2      42
KitchenQual        1
Functional         2
GarageType        76
GarageFinish      78
GarageQual        78
GarageCond        78
PoolQC          1456
SaleType           1
dtype: int64

In [ ]:
cat_cols_nulls_names = test_data[nulls.index].select_dtypes(
    include=['category' , 'object']).columns
                                                            
cat_cols_nulls_names

Index(['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'KitchenQual', 'Functional', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PoolQC', 'SaleType'],
      dtype='str')

In [8]:
cat_cols_nulls_names = cat_cols_nulls_names.difference([
    'MSZoning','Utilities', 'Exterior1st', 'Exterior2nd' ,
                'KitchenQual' , 'Functional' , 'SaleType'
                                                       ])
cat_cols_nulls_names

Index(['BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual',
       'GarageCond', 'GarageFinish', 'GarageQual', 'GarageType', 'PoolQC'],
      dtype='str')

In [9]:
test_data[cat_cols_nulls_names] = test_data[cat_cols_nulls_names].fillna('Missing')

In [10]:
mode_columns = ['MSZoning','Utilities', 'Exterior1st', 'Exterior2nd' ,
                'KitchenQual' , 'Functional' , 'SaleType']

for col in mode_columns :
    test_data[col] = test_data[col].fillna( train_data[col].mode()[0] )
    

In [11]:
nulls = test_data.isnull().sum()
nulls = nulls[nulls > 0]
nulls

LotFrontage     227
MasVnrArea       15
BsmtFinSF1        1
BsmtFinSF2        1
BsmtUnfSF         1
TotalBsmtSF       1
BsmtFullBath      2
BsmtHalfBath      2
GarageYrBlt      78
GarageCars        1
GarageArea        1
dtype: int64

##### Handling with Numerical Columns

In [12]:
num_cols_names = test_data[nulls.index].select_dtypes(include=np.number).isnull().sum()
num_cols_names

LotFrontage     227
MasVnrArea       15
BsmtFinSF1        1
BsmtFinSF2        1
BsmtUnfSF         1
TotalBsmtSF       1
BsmtFullBath      2
BsmtHalfBath      2
GarageYrBlt      78
GarageCars        1
GarageArea        1
dtype: int64

In [13]:
num_cols_names = test_data[nulls.index].select_dtypes(include=np.number).columns
num_cols_names = num_cols_names.drop('GarageYrBlt')
num_cols_names

Index(['LotFrontage', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
       'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars',
       'GarageArea'],
      dtype='str')

In [14]:
zero_cols = [
    'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'GarageYrBlt',
    'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea'
            ]

test_data[zero_cols] = test_data[zero_cols].fillna(0)

In [15]:
test_data['LotFrontage'] = test_data['LotFrontage'].fillna(
                           train_data['LotFrontage'].median() )

In [16]:
test_data.isnull().sum().sum()

np.int64(0)

In [17]:
test_data.to_csv('../datasets/processed/cleaned_test_data.csv' , index = False )